# OpenAI o3 Deep-Research Evaluation

Evaluate a selected OpenAI deep-research model on Calculus I multiple-choice questions. Responses are validated locally and checkpointed after each question so interrupted runs can resume.

## 1. Imports and configuration

In [ ]:
import json
import os
import re
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    precision_recall_fscore_support,
)

### Run configuration

Change only `MODEL_NAME` to select another deep-research model. The artifact directory and every generated filename update automatically.

In [ ]:
# Support launching Jupyter from either the repository root or notebooks/.
WORKING_DIRECTORY = Path.cwd()
PROJECT_ROOT = (
    WORKING_DIRECTORY.parent
    if WORKING_DIRECTORY.name == "notebooks"
    else WORKING_DIRECTORY
)

MODEL_NAME = "gpt-5.6-sol"
RUN_NAME = f"{MODEL_NAME}-o3-subsample"

DATA_PATH = PROJECT_ROOT / "data" / "test3_input_subset_seed42_numq1 (clean testing dataset).csv"
MODEL_OUTPUT_DIRECTORY = PROJECT_ROOT / "output" / MODEL_NAME
CHECKPOINT_PATH = MODEL_OUTPUT_DIRECTORY / f"{RUN_NAME}-checkpoint.json"
RESULTS_PATH = MODEL_OUTPUT_DIRECTORY / f"{RUN_NAME}-test-output.csv"
INCORRECT_RESULTS_PATH = (
    MODEL_OUTPUT_DIRECTORY / f"{RUN_NAME}-test-output-incorrect.csv"
)

MODEL_OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

## 2. OpenAI client and response schema

o3 deep research does not support API-enforced structured outputs. The model is instructed to return JSON, which is then parsed and validated locally with Pydantic.

In [ ]:
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"), timeout=3600)


class Calculus(BaseModel):
    """Validated response for one calculus question."""

    detailed_solution_explanation: str
    correct_option_choice_letter: str

### Few-shot example and instructions

In [ ]:
EXAMPLE_RESPONSE = '''
"Complete the square to write the equation in the standard form f(x)=a(x-h)^2+k. f(x)=2x^2+12x+25
                a) f(x)=2(x+3)^2+16
                b) f(x)=2(x-6)^2+7
                c) f(x)=2(x+3)^2+7
                d) f(x)=2(x-3)^2+16
                e) f(x)=2(x+3)^2-7
                f) None of the above."
    "detailed_solution_explanation": "To complete the square for f(x) = 2x² + 12x + 25, first factor out 2 from the terms involving x:\n\nf(x) = 2(x² + 6x) + 25.\n\nNext, complete the square inside the parentheses:\n\nx² + 6x = x² + 6x + 9 − 9 = (x + 3)² − 9.\n\nRewrite f(x):\n\nf(x) = 2[(x + 3)² − 9] + 25\n      = 2(x + 3)² − 18 + 25\n      = 2(x + 3)² + 7.\n",
    "correct_option_choice_letter": "c"
'''

SYSTEM_INSTRUCTIONS = (
    "Act as a Mathematician specializing in Calculus. Solve the given "
    "college-level Calculus 1 multiple-choice problem, using the given example "
    "as a reference. Provide a step-by-step explanation of the solution process, "
    "clearly outlining the reasoning behind each step. Choose the correct letter "
    "option for the given problem. Ensure that the solution is accurate, "
    "comprehensive, and easy to understand. Return only one valid JSON object "
    "with the keys detailed_solution_explanation and "
    "correct_option_choice_letter. Do not wrap the JSON in Markdown. "
    "Here is the example: "
    + EXAMPLE_RESPONSE
)

## 3. Load the dataset

In [ ]:
def load_dataset(data_path: Path) -> pd.DataFrame:
    """Load the evaluation dataset."""
    return pd.read_csv(data_path)


df = load_dataset(DATA_PATH)
print(f"Loaded {len(df)} questions from {DATA_PATH}")
df.head()

## 4. Response pipeline

Each successfully validated response is written to the model-specific checkpoint immediately.

In [ ]:
def load_checkpoint(checkpoint_path: Path) -> dict:
    """Load saved responses, or return an empty checkpoint."""
    if not checkpoint_path.exists():
        return {}
    with checkpoint_path.open("r", encoding="utf-8") as checkpoint_file:
        return json.load(checkpoint_file)


def save_checkpoint(checkpoint_path: Path, data: dict) -> None:
    """Persist responses as formatted JSON."""
    with checkpoint_path.open("w", encoding="utf-8") as checkpoint_file:
        json.dump(data, checkpoint_file, indent=4)

In [ ]:
def parse_calculus_response(response_text: str) -> dict:
    """Extract and validate the JSON object returned by the model."""
    start = response_text.find("{")
    end = response_text.rfind("}")
    if start == -1 or end == -1 or end < start:
        raise ValueError("The model response did not contain a JSON object.")

    payload = json.loads(response_text[start : end + 1])
    return Calculus.model_validate(payload).model_dump()


def query_openai(question: str, model: str = MODEL_NAME) -> dict:
    """Request and locally validate one deep-research calculus solution."""
    if not hasattr(client, "responses"):
        raise RuntimeError(
            "This notebook requires an OpenAI Python SDK version that supports "
            "client.responses. Upgrade the openai package and restart the kernel."
        )

    response = client.responses.create(
        model=model,
        instructions=SYSTEM_INSTRUCTIONS,
        input="Now solve the following question: " + question,
        tools=[{"type": "web_search_preview"}],
    )
    return parse_calculus_response(response.output_text)

In [ ]:
def collect_responses(
    dataframe: pd.DataFrame,
    checkpoint_path: Path,
    model: str = MODEL_NAME,
) -> dict:
    """Query unanswered rows and update the checkpoint incrementally."""
    checkpoint = load_checkpoint(checkpoint_path)

    for _, row in dataframe.iterrows():
        question_id = str(row["ID"])
        if question_id in checkpoint:
            print(f"Skipping ID {question_id}, already processed.")
            continue

        print(f"Processing ID {question_id}...")
        response = query_openai(row["Question"], model=model)
        checkpoint[question_id] = response
        save_checkpoint(checkpoint_path, checkpoint)

    save_checkpoint(checkpoint_path, checkpoint)
    print(f"Checkpoint saved to {checkpoint_path}")
    return checkpoint

In [ ]:
def add_model_responses(dataframe: pd.DataFrame, checkpoint: dict) -> pd.DataFrame:
    """Add answer choices and explanations from the checkpoint."""
    result = dataframe.copy()
    question_ids = result["ID"].astype(str)
    result["LLM Answer Choice (RAW)"] = question_ids.map(
        lambda question_id: checkpoint.get(question_id, {}).get(
            "correct_option_choice_letter", ""
        )
    )
    result["LLM Answer Explanation"] = question_ids.map(
        lambda question_id: checkpoint.get(question_id, {}).get(
            "detailed_solution_explanation", ""
        )
    )
    return result

### Run the response pipeline

> **Note:** This cell makes billable deep-research API calls only for questions absent from the checkpoint. Deep-research requests can be slow and comparatively expensive.

In [ ]:
df = df.head(10)  # Limit to first 10 rows for testing

checkpoint = collect_responses(df, CHECKPOINT_PATH)
df = add_model_responses(df, checkpoint)
df.head(10)

## 5. Normalize and evaluate answers

In [ ]:
def extract_letter_choice(text: str) -> str:
    """Return the first alphabetic character from a raw answer choice."""
    match = re.search(r"[a-zA-Z]", text)
    return match.group(0) if match else ""


def evaluate_answers(dataframe: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """Normalize predictions and calculate weighted classification metrics."""
    result = dataframe.copy()
    result["LLM Answer Choice"] = result["LLM Answer Choice (RAW)"].map(
        extract_letter_choice
    )
    result["Correct"] = (
        result["Correct Answer"] == result["LLM Answer Choice"]
    ).astype(int)

    y_true = result["Correct Answer"]
    y_pred = result["LLM Answer Choice"]
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "classification_report": classification_report(
            y_true, y_pred, zero_division=0
        ),
    }
    return result, metrics

In [ ]:
df, metrics = evaluate_answers(df)

print(metrics["classification_report"])
print(f"Accuracy:  {metrics['accuracy'] * 100:.2f}%")
print(f"Precision: {metrics['precision'] * 100:.2f}%")
print(f"Recall:    {metrics['recall'] * 100:.2f}%")
print(f"F1 Score:  {metrics['f1'] * 100:.2f}%")
df.head()

## 6. Export results

In [ ]:
incorrect_df = df.loc[df["Correct"] == 0].copy()

incorrect_df.to_csv(INCORRECT_RESULTS_PATH, index=False)
df.to_csv(RESULTS_PATH, index=False)

print(f"Saved {len(df)} results to {RESULTS_PATH}")
print(f"Saved {len(incorrect_df)} incorrect results to {INCORRECT_RESULTS_PATH}")